# 阶段四：最小真实训练实验

本 Notebook 使用冻结的真实 Reward 合同，在 CPU 上完成一个逻辑上共 5 步的最小训练实验。它先运行 4 步并保存检查点，然后执行一次未中断的第 5 步作为确定性参照；随后从第 4 步检查点恢复并重放第 5 步，逐项比较候选序列、训练统计、模型参数、`logZ`、优化器和随机状态。

所有产物写入已忽略的 `runs/real_minimal/<run_id>/`。请在干净内核中从上到下手动运行，不要将运行输出、数据或检查点提交 Git。

## 1. 环境与项目导入

In [ ]:
import gc
import json
import math
import os
import platform
import random
import sys
from copy import deepcopy
from dataclasses import asdict, is_dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents) if (path / 'factor_gfn').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('无法从当前目录向上找到 factor_gfn 项目根目录')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from factor_gfn.barra import STYLE_NAMES
from factor_gfn.gfn import (
    DEFAULT_REAL_REWARD_CONFIG,
    GFNConfig,
    GFNTrainer,
    ModelConfig,
    RealRewardProvider,
    SamplingConfig,
    SearchSpaceConfig,
    TrainingConfig,
    build_real_reward_data_context,
    write_run_metadata,
)

assert torch.device('cpu').type == 'cpu'
display(
    pd.DataFrame(
        [
            {
                'project_root': str(PROJECT_ROOT),
                'python': platform.python_version(),
                'numpy': np.__version__,
                'torch': torch.__version__,
                'device': 'cpu',
                'pid': os.getpid(),
            }
        ]
    )
)

## 2. 冻结本次最小实验配置

这些参数只是工程冒烟实验，不代表研报参数或正式训练配置。`max_steps=5` 是严格上限；第 5 步会分别以未中断和检查点恢复两种方式计算，用于确定性对照。

In [ ]:
DEVICE = 'cpu'
LOGICAL_TRAINING_STEPS = 5
CHECKPOINT_AFTER_STEP = 4

config = GFNConfig(
    search_space=SearchSpaceConfig(max_depth=5, max_nodes=12),
    model=ModelConfig(
        d_model=64,
        num_heads=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.0,
    ),
    sampling=SamplingConfig(temperature=1.0, greedy=False),
    reward=DEFAULT_REAL_REWARD_CONFIG,
    training=TrainingConfig(
        batch_size=2,
        learning_rate=1e-4,
        log_z_learning_rate=1e-3,
        max_steps=5,
        max_sampling_multiplier=10,
        seed=42,
    ),
)
assert config.training.max_steps == LOGICAL_TRAINING_STEPS
assert CHECKPOINT_AFTER_STEP == LOGICAL_TRAINING_STEPS - 1
assert config.reward.candidate_industry_neutralization is True
assert config.reward.barra_min_common_periods == 60
display(pd.json_normalize(config.manifest()['config']))

## 3. 全请求候选记录、JSON 与确定性比较工具

`RealRewardProvider.evaluation_records` 只记录真正执行的唯一计算。下面的薄包装器额外记录 Trainer 发出的每一次 Reward 请求，包括重复表达式和缓存命中，因此 `evaluations.jsonl` 覆盖整个实验而不只是最后一个 batch。

In [ ]:
def json_safe(value):
    if is_dataclass(value):
        return json_safe(asdict(value))
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(path, payload):
    path = Path(path).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    try:
        temporary.write_text(
            json.dumps(json_safe(payload), ensure_ascii=False, indent=2, allow_nan=False),
            encoding='utf-8',
        )
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def atomic_write_jsonl(path, records):
    path = Path(path).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    try:
        lines = [
            json.dumps(json_safe(record), ensure_ascii=False, allow_nan=False)
            for record in records
        ]
        temporary.write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


class RecordingRewardProvider:
    def __init__(self, delegate, *, branch):
        self.delegate = delegate
        self.branch = str(branch)
        self.phase = 'unassigned'
        self.records = []

    def manifest(self):
        return self.delegate.manifest()

    def fingerprint(self):
        return self.delegate.fingerprint()

    def evaluate(self, expression):
        assignment = self.delegate.evaluate(expression)
        metadata = deepcopy(assignment.metadata) if assignment.metadata else {}
        self.records.append(
            {
                'request_index': len(self.records) + 1,
                'branch': self.branch,
                'phase': self.phase,
                'formula': expression.to_formula(),
                'structural_hash': expression.structural_hash(),
                'prefix_token_ids': list(expression.to_prefix()),
                'node_count': expression.stats.node_count,
                'depth': expression.stats.depth,
                'valid': assignment.valid,
                'reward': assignment.reward,
                'log_reward': assignment.log_reward,
                'rejection_reason': assignment.rejection_reason,
                'provider_cache_hit': metadata.get('provider_cache_hit'),
                'metadata': metadata,
            }
        )
        return assignment


def clone_model_state(trainer):
    return {name: value.detach().cpu().clone() for name, value in trainer.model.state_dict().items()}


def capture_rng_state():
    return {
        'python': deepcopy(random.getstate()),
        'numpy': deepcopy(np.random.get_state()),
        'torch_cpu': torch.get_rng_state().clone(),
    }


def numpy_rng_equal(left, right):
    return (
        left[0] == right[0]
        and np.array_equal(left[1], right[1])
        and left[2:] == right[2:]
    )


def nested_equal(left, right):
    if isinstance(left, torch.Tensor) and isinstance(right, torch.Tensor):
        return torch.equal(left.cpu(), right.cpu())
    if isinstance(left, np.ndarray) and isinstance(right, np.ndarray):
        return np.array_equal(left, right)
    if isinstance(left, dict) and isinstance(right, dict):
        return left.keys() == right.keys() and all(
            nested_equal(left[key], right[key]) for key in left
        )
    if isinstance(left, (list, tuple)) and isinstance(right, type(left)):
        return len(left) == len(right) and all(
            nested_equal(a, b) for a, b in zip(left, right)
        )
    return left == right


def request_signature(records):
    return [
        (
            record['structural_hash'],
            record['valid'],
            record['reward'],
            record['log_reward'],
            record['rejection_reason'],
        )
        for record in records
    ]

## 4. 构建真实上下文、Provider 与 Trainer

该单元会加载真实数据并初始化 `float64` 解释器副本，但尚不训练。

In [ ]:
context = build_real_reward_data_context()
assert context.config.train_start == '2010-01-01'
assert context.config.train_end == '2018-12-31'
assert context.history_dates[-1] <= np.datetime64('2018-12-31')
assert context.manifest['industry']['universe_missing_count'] == 0
assert set(context.barra_long_short) == set(STYLE_NAMES)

real_provider = RealRewardProvider(context, config.reward)
recording_provider = RecordingRewardProvider(real_provider, branch='main')
trainer = GFNTrainer(config, recording_provider, device=DEVICE)
run_dir = PROJECT_ROOT / 'runs' / 'real_minimal' / trainer.run_id
run_dir.mkdir(parents=True, exist_ok=False)
checkpoint_step4_path = run_dir / 'checkpoint_step4.pt'
checkpoint_resumed_path = run_dir / 'checkpoint_resumed_step5.pt'
run_metadata_path = run_dir / 'run_metadata.json'
training_stats_path = run_dir / 'training_stats.json'
evaluations_path = run_dir / 'evaluations.jsonl'
best_candidate_path = run_dir / 'best_candidate.json'
determinism_path = run_dir / 'determinism_report.json'
experiment_manifest_path = run_dir / 'experiment_manifest.json'

initial_model_state = clone_model_state(trainer)
initial_log_z = float(trainer.tb_loss.log_z.detach())
assert trainer.step == 0
assert trainer.optimizer_step == 0
assert initial_log_z == 0.0

display(
    pd.DataFrame(
        [
            {
                'run_id': trainer.run_id,
                'run_dir': str(run_dir),
                'context_fingerprint': context.fingerprint,
                'provider_fingerprint': recording_provider.fingerprint(),
                'rebalance_periods': int(context.rebalance_indices.size),
                'first_rebalance_date': context.manifest['calendar']['first_rebalance_date'],
                'last_rebalance_date': context.manifest['calendar']['last_rebalance_date'],
            }
        ]
    )
)

## 5. 运行前 4 步并保存检查点

第 4 步结束时同时要求 `trainer.step == 4` 和 `trainer.optimizer_step == 4`。如果前四步出现跳过更新，本单元会停止，不会伪装成已完成四次参数更新。

In [ ]:
first_four_stats = []
for expected_step in range(1, CHECKPOINT_AFTER_STEP + 1):
    recording_provider.phase = f'train_step_{expected_step}'
    stats = trainer.train_step()
    first_four_stats.append(stats)
    assert trainer.step == expected_step
    display(pd.DataFrame([asdict(stats)]))
    atomic_write_jsonl(evaluations_path, recording_provider.records)
    atomic_write_json(
        training_stats_path,
        {
            'schema': 'factor_gfn.real_minimal_training_stats.v1',
            'complete': False,
            'logical_history': [asdict(item) for item in trainer.history],
        },
    )

assert trainer.step == CHECKPOINT_AFTER_STEP == 4
assert trainer.optimizer_step == CHECKPOINT_AFTER_STEP == 4

trainer.save_checkpoint(checkpoint_step4_path)
assert checkpoint_step4_path.is_file()

step4_model_state = clone_model_state(trainer)
step4_log_z = trainer.tb_loss.log_z.detach().cpu().clone()
step4_optimizer_state = deepcopy(trainer.optimizer.state_dict())
step4_rng_state = capture_rng_state()
step4_request_count = len(recording_provider.records)
print('checkpoint:', checkpoint_step4_path)
print('current_step:', trainer.step, '| optimizer_step:', trainer.optimizer_step)

## 6. 未中断地运行第 5 步，保存确定性参照

这一步形成“一次性跑完 5 步”的参照结果。随后会删除该 Trainer，并从第 4 步检查点重放相同的第 5 步。

In [ ]:
recording_provider.phase = 'continuous_reference_step_5'
continuous_step5_stats = trainer.train_step()
assert trainer.step == LOGICAL_TRAINING_STEPS == 5

continuous_step5_records = deepcopy(
    recording_provider.records[step4_request_count:]
)
main_branch_records = deepcopy(recording_provider.records)
expected_history = deepcopy(trainer.history)
expected_model_state = clone_model_state(trainer)
expected_log_z = trainer.tb_loss.log_z.detach().cpu().clone()
expected_optimizer_state = deepcopy(trainer.optimizer.state_dict())
expected_rng_state = capture_rng_state()
expected_optimizer_step = trainer.optimizer_step
display(pd.DataFrame([asdict(continuous_step5_stats)]))

## 7. 释放原 Provider，创建新 Trainer 并恢复第 4 步检查点

使用新的 `RealRewardProvider` 可以避免把旧 Provider 的进程内缓存误当作检查点状态。检查点加载后先核对第 4 步模型、`logZ`、优化器和随机状态，再运行第 5 步。

In [ ]:
del trainer, recording_provider, real_provider
gc.collect()

restored_real_provider = RealRewardProvider(context, config.reward)
restored_provider = RecordingRewardProvider(
    restored_real_provider,
    branch='deterministic_replay',
)
restored_trainer = GFNTrainer(config, restored_provider, device=DEVICE)
restored_metadata = restored_trainer.load_checkpoint(checkpoint_step4_path)

checkpoint_restore_checks = {
    'current_step_is_4': restored_trainer.step == 4,
    'optimizer_step_is_4': restored_trainer.optimizer_step == 4,
    'model_state_exact': nested_equal(
        step4_model_state, clone_model_state(restored_trainer)
    ),
    'log_z_exact': torch.equal(
        step4_log_z, restored_trainer.tb_loss.log_z.detach().cpu()
    ),
    'optimizer_state_exact': nested_equal(
        step4_optimizer_state, restored_trainer.optimizer.state_dict()
    ),
    'python_rng_exact': random.getstate() == step4_rng_state['python'],
    'numpy_rng_exact': numpy_rng_equal(
        np.random.get_state(), step4_rng_state['numpy']
    ),
    'torch_rng_exact': torch.equal(
        torch.get_rng_state(), step4_rng_state['torch_cpu']
    ),
    'run_id_restored': restored_trainer.run_id == restored_metadata['run_id'],
}
assert all(checkpoint_restore_checks.values()), checkpoint_restore_checks
display(pd.DataFrame([checkpoint_restore_checks]))

## 8. 恢复后重放第 5 步并进行逐位比较

In [ ]:
restored_provider.phase = 'restored_step_5'
restored_step5_stats = restored_trainer.train_step()
assert restored_trainer.step == LOGICAL_TRAINING_STEPS == 5

restored_rng_state = capture_rng_state()
determinism_checks = {
    'candidate_request_sequence_exact': (
        request_signature(continuous_step5_records)
        == request_signature(restored_provider.records)
    ),
    'training_stats_exact': continuous_step5_stats == restored_step5_stats,
    'history_exact': expected_history == restored_trainer.history,
    'model_state_exact': nested_equal(
        expected_model_state, clone_model_state(restored_trainer)
    ),
    'log_z_exact': torch.equal(
        expected_log_z, restored_trainer.tb_loss.log_z.detach().cpu()
    ),
    'optimizer_state_exact': nested_equal(
        expected_optimizer_state, restored_trainer.optimizer.state_dict()
    ),
    'python_rng_exact': restored_rng_state['python'] == expected_rng_state['python'],
    'numpy_rng_exact': numpy_rng_equal(
        restored_rng_state['numpy'], expected_rng_state['numpy']
    ),
    'torch_rng_exact': torch.equal(
        restored_rng_state['torch_cpu'], expected_rng_state['torch_cpu']
    ),
}
assert all(determinism_checks.values()), determinism_checks
assert restored_trainer.optimizer_step == expected_optimizer_step
display(pd.DataFrame([determinism_checks]))
display(pd.DataFrame([asdict(restored_step5_stats)]))

## 9. 汇总全部候选并原子写入运行产物

`main_branch_records` 是逻辑 5 步训练的全部 Reward 请求；`restored_provider.records` 是确定性重放请求。两者都写入 JSONL，并通过 `branch`、`phase` 区分。最佳候选只从逻辑主分支选择，避免把重放重复计入。

In [ ]:
all_evaluation_records = main_branch_records + deepcopy(restored_provider.records)
valid_main_records = [
    record
    for record in main_branch_records
    if record['valid'] and record['reward'] is not None
]
best_candidate = (
    max(valid_main_records, key=lambda record: record['reward'])
    if valid_main_records
    else None
)

restored_trainer.save_checkpoint(checkpoint_resumed_path)
write_run_metadata(run_metadata_path, restored_trainer)
atomic_write_jsonl(evaluations_path, all_evaluation_records)
atomic_write_json(
    training_stats_path,
    {
        'schema': 'factor_gfn.real_minimal_training_stats.v1',
        'complete': True,
        'logical_history': [asdict(item) for item in restored_trainer.history],
        'continuous_reference_step5': asdict(continuous_step5_stats),
        'restored_step5': asdict(restored_step5_stats),
    },
)
atomic_write_json(
    best_candidate_path,
    {
        'schema': 'factor_gfn.real_minimal_best_candidate.v1',
        'candidate': best_candidate,
    },
)
atomic_write_json(
    determinism_path,
    {
        'schema': 'factor_gfn.real_minimal_determinism.v1',
        'checkpoint_restore': checkpoint_restore_checks,
        'step5_replay': determinism_checks,
    },
)
artifact_paths = [
    checkpoint_step4_path,
    checkpoint_resumed_path,
    run_metadata_path,
    training_stats_path,
    evaluations_path,
    best_candidate_path,
    determinism_path,
    experiment_manifest_path,
]
experiment_manifest = {
    'schema': 'factor_gfn.real_minimal_experiment.v1',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_id': restored_trainer.run_id,
    'run_dir': str(run_dir),
    'logical_training_steps': LOGICAL_TRAINING_STEPS,
    'checkpoint_after_step': CHECKPOINT_AFTER_STEP,
    'deterministic_replay_extra_step': True,
    'main_reward_requests': len(main_branch_records),
    'replay_reward_requests': len(restored_provider.records),
    'unique_main_expressions': len(
        {record['structural_hash'] for record in main_branch_records}
    ),
    'valid_main_requests': len(valid_main_records),
    'artifacts': [str(path) for path in artifact_paths],
}
atomic_write_json(experiment_manifest_path, experiment_manifest)
assert all(path.is_file() for path in artifact_paths)
display(pd.DataFrame({'artifact': [str(path) for path in artifact_paths]}))

## 10. 训练统计、最佳候选与最终验收

若拒绝率超过 80%，这里只报告并要求分析，不会降低 60 期门槛、关闭行业中性化或扩大训练规模。

In [ ]:
history_frame = pd.DataFrame([asdict(item) for item in restored_trainer.history])
display(
    history_frame[
        [
            'step', 'optimizer_step', 'loss', 'reward_mean', 'reward_median',
            'log_z', 'gradient_norm', 'effective_batch_size',
            'batch_rejection_rate', 'skipped_update',
            'expression_unique_rate', 'trajectory_length_mean',
            'trajectory_length_max', 'policy_entropy_mean',
            'policy_entropy_normalized_mean', 'illegal_action_rate',
        ]
    ]
)

changed_model_tensors = [
    name
    for name, value in restored_trainer.model.state_dict().items()
    if not torch.equal(value.detach().cpu(), initial_model_state[name])
]
non_skipped = history_frame[~history_frame['skipped_update']]
high_rejection = history_frame[history_frame['batch_rejection_rate'] > 0.80]
final_log_z = float(restored_trainer.tb_loss.log_z.detach())
acceptance_checks = {
    'five_logical_steps_recorded': len(history_frame) == 5 and restored_trainer.step == 5,
    'at_least_three_optimizer_updates': restored_trainer.optimizer_step >= 3,
    'transformer_changed': bool(changed_model_tensors),
    'log_z_changed_and_finite': math.isfinite(final_log_z) and final_log_z != initial_log_z,
    'finite_non_skipped_loss': bool(
        len(non_skipped) > 0 and np.isfinite(non_skipped['loss'].to_numpy(dtype=float)).all()
    ),
    'illegal_action_rate_zero': bool((history_frame['illegal_action_rate'] == 0.0).all()),
    'checkpoint_restore_exact': all(checkpoint_restore_checks.values()),
    'restored_step5_exact': all(determinism_checks.values()),
    'all_artifacts_exist': all(path.is_file() for path in artifact_paths),
}
assert all(acceptance_checks.values()), acceptance_checks
display(pd.DataFrame([acceptance_checks]))

summary = {
    'run_dir': str(run_dir),
    'optimizer_updates': restored_trainer.optimizer_step,
    'skipped_updates': int(history_frame['skipped_update'].sum()),
    'high_rejection_steps': int(len(high_rejection)),
    'main_reward_requests': len(main_branch_records),
    'unique_main_expressions': len(
        {record['structural_hash'] for record in main_branch_records}
    ),
    'valid_main_requests': len(valid_main_records),
    'best_formula': best_candidate['formula'] if best_candidate else None,
    'best_reward': best_candidate['reward'] if best_candidate else None,
    'best_barra_correlations': (
        best_candidate['metadata']['reward_result']['barra_correlations']
        if best_candidate else None
    ),
    'best_neutralization_skipped_dates': (
        best_candidate['metadata']['reward_result']['neutralization_skipped_dates']
        if best_candidate else None
    ),
    'best_neutralization_skipped_rate': (
        best_candidate['metadata']['reward_result']['neutralization_skipped_rate']
        if best_candidate else None
    ),
    'best_dominant_barra_factor': (
        best_candidate['metadata']['reward_result']['dominant_barra_factor']
        if best_candidate else None
    ),
    'best_dominant_barra_correlation': (
        best_candidate['metadata']['reward_result']['dominant_barra_correlation']
        if best_candidate else None
    ),
}
display(pd.DataFrame([summary]))
if len(high_rejection):
    print('警告：存在拒绝率超过 80% 的步骤，请先分析 evaluations.jsonl 中的拒绝原因。')
print('运行目录：', run_dir)

## 11. 手工运行后的回传内容

请回传最后三个表格以及运行目录中的 `training_stats.json`、`determinism_report.json`、`best_candidate.json` 和 `experiment_manifest.json`。如果验收失败或拒绝率偏高，先分析当前产物，不自动修改 Reward、行业中性化、60期门槛或训练规模。